# EuroBonds: Sparse-Robust vs Sparse (out-of-sample performance)

This notebook reproduces the empirical result that on a **wide volatility-dispersion** universe
(EuroBondsRet, N=63, p90/p10 vol ratio ~14.5x) the **Sparse-Robust** portfolio (CP-RMVP, $\gamma>0$)
beats the plain **Sparse** portfolio ($\gamma=0$) out of sample, by tilting toward low-volatility
assets (consistent with the low-volatility anomaly).

Setup: rolling window (in-sample W, out-of-sample block H), warm-start drop 0.5 for tractable BnB,
BLAS threads pinned to 1 (clean CPU timing). All portfolios use the SAME pipeline as the paper
(`build_problem`, `solve_model`, `oos_eval` from `referee_experiments.py`).

Metrics per (beta, gamma): OOS excess Sharpe (pooled + per-window paired t of robust-minus-sparse),
average support size, and wall / CPU solve time.

In [ ]:
import os
# Pin BLAS/OpenMP threads BEFORE importing numpy -> clean, reproducible timing.
for _v in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS',
           'NUMEXPR_NUM_THREADS','VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(_v, '1')
import sys, glob
from types import SimpleNamespace
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Make sure the rmvp package dir is importable (adjust if running elsewhere).
RMVP_DIR = os.getcwd()
if RMVP_DIR not in sys.path: sys.path.insert(0, RMVP_DIR)

from referee_experiments import build_problem, solve_model, oos_eval
from eurobonds_experiment import load_returns, run_grid

DATA = os.path.join('datasets','EuroBondsRet.xlsx')
R = load_returns(DATA)
vols = R.std(axis=0)
print(f'EuroBonds: {R.shape[0]} days x {R.shape[1]} assets')
print(f'daily vol  p10={np.percentile(vols,10):.4f}  median={np.median(vols):.4f}  '
      f'p90={np.percentile(vols,90):.4f}  -> dispersion {np.percentile(vols,90)/np.percentile(vols,10):.1f}x')

## 1. Quick reproduction (small grid)

A reduced grid you can run in a few minutes to verify the effect. `max_windows` caps the number of
rebalances; set it to 0 (and widen the grid) to reproduce the full result (slower).

In [ ]:
args = SimpleNamespace(
    window=60, test=15, r_c=0.0002, tr_factor=1.05,
    betas=[2e-6, 1e-6], gammas=[0.0, 0.05, 0.10, 0.15],
    drop=0.5, timeout=90, max_windows=25,   # <- set 0 for the full backtest
)
rows = run_grid(R, args)
df = pd.DataFrame(rows)
cols = ['beta','gamma','Sparse_supp','SparseRobust_supp',
        'Markowitz_pooledSharpe','Sparse_pooledSharpe','SparseRobust_pooledSharpe',
        'paired_meanDiff','paired_t','Sparse_wall_s','SparseRobust_wall_s','SparseRobust_cpu_s']
df[cols].round(3)

## 2. Pooled OOS Sharpe: Sparse vs Sparse-Robust across gamma

For each beta, the robust Sharpe should rise above the (flat) sparse Sharpe as gamma increases.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
for b, g in df.groupby('beta'):
    ax.plot(g['gamma'], g['SparseRobust_pooledSharpe'], 'o-', label=f'Robust beta={b:.0e}')
    ax.axhline(g['Sparse_pooledSharpe'].iloc[0], ls='--', alpha=0.5,
               label=f'Sparse beta={b:.0e}')
ax.set_xlabel('gamma (robustness radius)'); ax.set_ylabel('pooled OOS excess Sharpe')
ax.set_title('EuroBonds: robust tilt raises OOS Sharpe over sparse'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 3. Mechanism: robust selects lower-volatility assets

For a fixed (beta, gamma) we compare the average volatility of the assets each model selects,
relative to the universe median. Robust should pick BELOW-median-vol assets; sparse ABOVE.

In [ ]:
raw_vol = R.std(axis=0); med = np.median(raw_vol)
W, H, beta, gamma = 60, 15, 1e-6, 0.10
starts = list(range(W, R.shape[0]-H, H))[-40:]
res = {'Sparse':{'supp':[], 'vol':[]}, 'SparseRobust':{'supp':[], 'vol':[]}}
for t in starts:
    D, tau, tb = build_problem(R[t-W:t], 0.0002, 1.05)
    for mdl, gg in [('Sparse',0.0), ('SparseRobust',gamma)]:
        s = solve_model(mdl, D, tau, tb, gg, beta, 0.5, 90)
        if s['status']!='ok': continue
        idx = np.where(np.abs(s['x_full'])>1e-8)[0]
        res[mdl]['supp'].append(len(idx)); res[mdl]['vol'].append(raw_vol[idx].mean() if len(idx) else np.nan)
for m in ['Sparse','SparseRobust']:
    print(f"{m:13s} avg_support={np.mean(res[m]['supp']):.1f}  "
          f"avg_selected_vol={np.nanmean(res[m]['vol']):.4f}  "
          f"({np.nanmean(res[m]['vol'])/med:.2f}x median)")

## 4. Full precomputed grid (from `eurobonds_experiment.py --tag full_`)

Loads the complete backtest table (all windows, betas {5e-6,2e-6,1e-6}, gammas {0,0.05,0.10,0.15})
produced by the standalone script, including wall/CPU timing.

In [ ]:
files = sorted(glob.glob('eurobonds_exp_full_*.xlsx'))
if files:
    full = pd.read_excel(files[-1])
    show = ['beta','gamma','Sparse_supp','SparseRobust_supp','Markowitz_pooledSharpe',
            'Sparse_pooledSharpe','SparseRobust_pooledSharpe','paired_t',
            'Sparse_wall_s','SparseRobust_wall_s','SparseRobust_cpu_s']
    display(full[show].round(3))
else:
    print('Run:  python3 eurobonds_experiment.py --window 60 --test 15 '
          '--betas 5e-6 2e-6 1e-6 --gammas 0.0 0.05 0.10 0.15 --drop 0.5 --tag full_')